## Objective: Getting the data for similar non-profits
#### In this notebook, we will work on extracting data from prorepublica, and the IRS form 990 to help us identify the most important fields for our analyses.
Our main goal is to understand how the non-profits that are similar to Pacific Beach Coalition are being funded. How their operations are ran (employees, employee compensation, grant funding, program services, revenue etc.). This will then enable us to come up with specific strategies to help them achieve their funding and operations goal. 

### 1. Webscraping form 990 Data points from Prorepublica
To fetch Form 990 filings for the specified EINs (Employer Identification Numbers) for the years 2019–2023;
1. `fetch_filings Function` - This function retrieves filings for a specific EIN for the years 2019–2023 from the "filings_with_data" 
2. section of the API response.
    - List of EINs and Years - All EINs gotten from GuideHouse are included in this list. The years list includes 2019–2023.
3. The results are saved in a JSON file called `filings_results.json`.

In [1]:
import requests
import json

def search_nonprofit(query, page=0):
    """
    Search for nonprofits matching the query using the ProPublica Nonprofit Explorer API.

    :param query: A string query to search for.
    :param page: The page number to fetch (default: 0).
    :return: List of nonprofits matching the search criteria.
    """
    base_url = "https://projects.propublica.org/nonprofits/api/v2/search.json"
    params = {
        "q": query,
        "page": page
    }
    response = requests.get(base_url, params=params)

    if response.status_code == 200:
        data = response.json()
        return data["organizations"]
    else:
        print(f"Error: {response.status_code}")
        return []

def get_nonprofit_details(ein):
    """
    Fetch detailed information about a nonprofit using its EIN.

    :param ein: Employer Identification Number (EIN) of the nonprofit.
    :return: Details of the nonprofit and its filings.
    """
    base_url = f"https://projects.propublica.org/nonprofits/api/v2/organizations/{ein}.json"
    response = requests.get(base_url)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        return {}

def fetch_filings(ein, years):
    """
    Fetch filings for a given EIN and a range of years.

    :param ein: Employer Identification Number (EIN) of the nonprofit.
    :param years: List of years to fetch filings for.
    :return: A dictionary with year as key and filing details as value.
    """
    filings = {}
    details = get_nonprofit_details(ein)
    if "filings_with_data" in details:
        for filing in details["filings_with_data"]:
            tax_year = filing.get("tax_prd_yr")
            if tax_year in years:
                filings[tax_year] = filing
    return filings

def main():
    print("Fetching Form 990 filings for specific EINs and years (2019-2023)...")

    eins = [
        "91-1767292", "33-0647946", "84-4780735", "45-4329874", "83-3305529",
        "33-0587567", "94-2745941", "33-0145660", "94-2665367", "20-1832617",
        "47-5161428", "94-2216915", "23-7213237", "85-3432174", "33-0974992",
        "94-1693226", "22-3902362", "95-6377791", "33-0537412", "45-3042628",
        "95-2496099", "45-2639830", "95-3273023", "68-0120240", "95-2566791",
        "77-0565183", "82-4594246", "83-1159078", "01-0777856", "38-3891081",
        "94-2951488", "94-2788588", "88-1650309", "43-2050242", "30-0358349",
        "46-5112972","83-1241456"
    ]

    years = [2019, 2020, 2021, 2022, 2023]
    results = {}

    for ein in eins:
        print(f"Fetching filings for EIN: {ein}")
        filings = fetch_filings(ein, years)
        if filings:
            results[ein] = filings
        else:
            print(f"No filings found for EIN {ein} in the specified years.")

    # Save or display results
    with open("filings_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Results saved to filings_results.json")

if __name__ == "__main__":
    main()


Fetching Form 990 filings for specific EINs and years (2019-2023)...
Fetching filings for EIN: 91-1767292
Fetching filings for EIN: 33-0647946
Fetching filings for EIN: 84-4780735
Fetching filings for EIN: 45-4329874
Fetching filings for EIN: 83-3305529
Fetching filings for EIN: 33-0587567
Fetching filings for EIN: 94-2745941
Fetching filings for EIN: 33-0145660
Fetching filings for EIN: 94-2665367
Fetching filings for EIN: 20-1832617
Fetching filings for EIN: 47-5161428
Fetching filings for EIN: 94-2216915
Fetching filings for EIN: 23-7213237
Fetching filings for EIN: 85-3432174
Fetching filings for EIN: 33-0974992
Fetching filings for EIN: 94-1693226
Fetching filings for EIN: 22-3902362
Fetching filings for EIN: 95-6377791
Fetching filings for EIN: 33-0537412
Fetching filings for EIN: 45-3042628
Fetching filings for EIN: 95-2496099
Fetching filings for EIN: 45-2639830
Fetching filings for EIN: 95-3273023
Fetching filings for EIN: 68-0120240
Fetching filings for EIN: 95-2566791
Fetchi

### 2. Parsing JSON file gotten from prorepublica
This script parses the file that was webscraped from prorepublica's API to extract the fields that will be most relevant for further analysis such as (compensation, total revenue per year, expenses, other wages, grants etc. )

In [2]:
import json
import pandas as pd

# Load the JSON data
file_path = 'filings_results.json'
with open(file_path, 'r') as f:
    filings_data = json.load(f)

# Define columns for the DataFrame
columns = [
    "EIN", "Year", "Total Revenue", "Total Expenses",
    "Grants and Contributions", "Other Revenue", 
    "Other Salaries and Wages", "Compensation of Officers",
    "Net Assets at End", "Total Grants Proportion"
]

# Initialize rows for the DataFrame
rows = []

# Parse JSON and structure data for each filing
for ein, filings in filings_data.items():
    for year, details in filings.items():
        total_revenue = details.get('totrevenue', 0)
        total_expenses = details.get('totfuncexpns', 0)
        grants_contributions = details.get('totcntrbgfts', 0)
        other_revenue = details.get('miscrevtot11e', 0) + details.get('invstmntinc', 0) + details.get('totprgmrevnue', 0)
        other_salaries_wages = details.get('othrsalwages', 0)
        compensation_officers = details.get('compnsatncurrofcr', 0)
        net_assets_end = details.get('totnetassetend', 0)
        total_grants_proportion = (grants_contributions / total_revenue) if total_revenue > 0 else 0

        rows.append([
            ein, details.get('tax_prd_yr'), total_revenue, total_expenses,
            grants_contributions, other_revenue, 
            other_salaries_wages, compensation_officers,
            net_assets_end, total_grants_proportion
        ])

# Create DataFrame
df = pd.DataFrame(rows, columns=columns)
# Rounding the "Total Grants Proportion" column to 2 decimal places
df["Total Grants Proportion"] = df["Total Grants Proportion"].round(2)

df


,EIN,Year,Total Revenue,Total Expenses,Grants and Contributions,Other Revenue,Other Salaries and Wages,Compensation of Officers,Net Assets at End,Total Grants Proportion
0,91-1767292,2022,830209,813807,459662,370547,451558,0,318247,0.55
1,91-1767292,2021,514750,656805,382453,132297,402741,0,259099,0.74
2,91-1767292,2020,648231,716454,442842,205389,296667,86154,240770,0.68
3,91-1767292,2019,772717,824310,412455,360262,297074,84462,257007,0.53
4,33-0647946,2021,1128832,841936,1128928,4302,447050,155000,1609104,1.00
...,...,...,...,...,...,...,...,...,...,...
128,46-5112972,2019,115019,164778,0,0,0,0,0,0.00
129,83-1241456,2022,206452,153081,119405,87047,0,43191,207678,0.58
130,83-1241456,2021,213425,164777,138711,74714,0,28764,151781,0.65
131,83-1241456,2020,138790,163818,0,0,0,0,0,0.00


In [3]:
df.to_csv("grants.csv", index=False)

### 3. Convert Scanned image PDFS to text-based searchable pdfs

The script below automatically converts multiple scanned PDFs (in our case 120 Form 990 files) into searchable, text-based PDFs using Tesseract OCR library in Python. This will allow us to then parse the text-based pdf to further extract fields such as compensation of officers and amount.

1. Input and Output directory setup.
    - Input Directory (input_dir): This folder contains the original scanned PDFs that need to be processed.
    - Output Directory (output_dir): This folder is where the searchable PDFs will be saved after processing.
    - Page conversion using pdf2image (convert_From_path) - converts each page of a pdf into an image. This is necessary because OCR software, like Tesseract, can only process images, not PDFs directly.

2. Page Conversion Using pdf2.image
    - The `convert_to_searchable_pdf` function takes the PDF path as input and generates a list of images, one for each page of the PDF. 
    - `Image.Resampling.LANCZOS` - Before passing the images to OCR, we use `PIL` from `Image` to resize them. This ensures the images are downscaled and optimized for smaller file sizes before the searchable text layer is added. 
3. Optical Character Recognition (OCR) with pytesseract
    - This `pytesseract.image_to_pdf_or_hocr(img, extension='pdf', config='--pdf-renderer 1 --tessedit_create_pdf 1)` processes - each image (page of the scanned PDF) with Tesseract OCR. 
    - `--pdf-renderer 1`- Uses Tesseract’s PDF renderer, which is efficient for searchable PDFs.
    - `--tessedit_create_pdf 1` - Ensures a text layer is created and embedded in the PDF.
    - Then outputs a PDF binary that contains the original image overlaid with a text layer, making it searchable.
4. Creating a searchable pdf
    - The OCR ouput for each page is returned as a binary PDF file. 
    - This binary data is wrapped in a BytesIO object to simulate a file-like object, which is compatible with the PdfWriter.
    - `PdfWriter.append()` appends each OCR-processed page (wrapped as a PDF stream) to a new PDF document. This ensures that the searchable PDF combines all the OCR-processed pages into a single file.
5. Saving the Final Output as a searchable pdf
    - Once all the pages of a PDF are processed, the `PdfWriter` writes the combined searchable PDF to the specified output path.
6. Processing Multiple PDFs
    - The script loops through all the PDF files in the input directory.
    - For each file:
        - It reads the original scanned PDF.
        - Processes each page to make it searchable.
        - Saves the resulting searchable PDF in the output directory, prefixed with searchable_ for easy identification.

In [2]:
import os
from pdf2image import convert_from_path
import pytesseract
from pytesseract import Output
from PIL import Image
from PyPDF2 import PdfWriter
from io import BytesIO

# Path to Tesseract executable (adjust this for your system)
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

# Input and output directories
input_dir = "form_990_pdfs"  # Folder containing scanned PDFs
output_dir = "form_990_searchable_pdfs"  # Folder to save searchable PDFs

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

def convert_to_searchable_pdf(pdf_path, output_path):
    # Convert PDF pages to images
    images = convert_from_path(pdf_path)
    
    # Create a new searchable PDF
    pdf_writer = PdfWriter()
    
    for img in images:
         # first resize the image to reduce resolution to 50% of original size
        #img = img.resize((int(img.width / 2), int(img.height / 2)), Image.Resampling.LANCZOS)

        # Perform OCR on each image and get pdf data
        pdf_binary = pytesseract.image_to_pdf_or_hocr(img, extension='pdf')
        
        # Read the binary PDF data into a file-like object
        pdf_stream = BytesIO(pdf_binary)
        
        # Append the OCR layer to the PDF writer
        pdf_writer.append(pdf_stream)
    
    # Save the new searchable PDF
    with open(output_path, "wb") as f_out:
        pdf_writer.write(f_out)

# Process all PDFs in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, f"searchable_{filename}")
        print(f"Processing {filename}...")
        convert_to_searchable_pdf(input_path, output_path)
        print(f"Saved searchable PDF to {output_path}")

Processing 911767292_202006_990_2021040617900423.pdf...
Saved searchable PDF to form_990_searchable_pdfs/searchable_911767292_202006_990_2021040617900423.pdf
Processing 911767292_202206_990_2023031021079290.pdf...
Saved searchable PDF to form_990_searchable_pdfs/searchable_911767292_202206_990_2023031021079290.pdf
Processing 911767292_201906_990_2020020317099728.pdf...
Saved searchable PDF to form_990_searchable_pdfs/searchable_911767292_201906_990_2020020317099728.pdf
Processing 911767292_202206_990_2024011622239623.pdf...
Saved searchable PDF to form_990_searchable_pdfs/searchable_911767292_202206_990_2024011622239623.pdf


#### 3a. Script to Compress All PDFs in a Folder
This script uses PyMuPDF (fitz) to compress all the PDFs in a specified directory. The script loops through all PDF files in the input folder.
- For each file: It opens the PDF using fitz.open.
- Compresses the content streams and saves the result in the output folder with a compressed_ prefix.
- For file handlling, Non-PDF files are skipped. 
- If the output folder doesn’t exist, it’s created automatically.
- All compressed PDFs will be saved in the output_folder with filenames prefixed by compressed_.

In [6]:
import os
import fitz  # PyMuPDF

def compress_pdf(input_pdf, output_pdf):
    """
    Compress a single PDF using PyMuPDF.
    """
    doc = fitz.open(input_pdf)
    doc.save(output_pdf, garbage=4, deflate=True)  # Compress streams and garbage collect
    doc.close()

def compress_all_pdfs_in_folder(input_folder, output_folder):
    """
    Compress all PDFs in a folder.
    """
    os.makedirs(output_folder, exist_ok=True)  # Ensure output folder exists

    for filename in os.listdir(input_folder):
        if filename.endswith(".pdf"):  # Process only PDF files
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, f"compressed_{filename}")
            print(f"Compressing {filename}...")
            compress_pdf(input_path, output_path)
            print(f"Compressed file saved as {output_path}")

# Specify the input and output folders
input_folder = "form_990_searchable_pdfs"  # Folder with searchable PDFs
output_folder = "form_990_compressed_pdfs" # Folder to save compressed PDFs

# Compress all PDFs in the folder
compress_all_pdfs_in_folder(input_folder, output_folder)


Compressing searchable_330647946_202112_990_2023051221225435.pdf...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_330647946_202112_990_2023051221225435.pdf
Compressing searchable_330647946_201912_990_2021022217742860.pdf...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_330647946_201912_990_2021022217742860.pdf
Compressing searchable_330647946_202012_990_2022010519394111.pdf...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_330647946_202012_990_2022010519394111.pdf
Compressing searchable_911767292_202206_990_2023031021079290.pdf...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_911767292_202206_990_2023031021079290.pdf
Compressing searchable_911767292_202306_990_2024031222315919.pdf...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_911767292_202306_990_2024031222315919.pdf
Compressing searchable_330647946_202212_990_2024031222315271.pdf...
Compressed file saved 

#### 3b. Use Ghostscript for Batch Compression

In [8]:
import os

def compress_with_ghostscript(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for filename in os.listdir(input_folder):
        if filename.endswith(".pdf"):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, f"compressed_{filename}")
            print(f"Compressing {filename} with Ghostscript...")
            os.system(f"gs -sDEVICE=pdfwrite -dCompatibilityLevel=1.4 -dPDFSETTINGS=/ebook -dNOPAUSE -dQUIET -dBATCH -sOutputFile={output_path} {input_path}")
            print(f"Compressed file saved as {output_path}")

# Specify input and output folders
input_folder = "form_990_searchable_pdfs"  # Folder with searchable PDFs
output_folder = "form_990_compressed_pdfs" # Folder to save compressed PDFs

compress_with_ghostscript(input_folder, output_folder)


Compressing searchable_330647946_202112_990_2023051221225435.pdf with Ghostscript...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_330647946_202112_990_2023051221225435.pdf
Compressing searchable_330647946_201912_990_2021022217742860.pdf with Ghostscript...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_330647946_201912_990_2021022217742860.pdf
Compressing searchable_330647946_202012_990_2022010519394111.pdf with Ghostscript...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_330647946_202012_990_2022010519394111.pdf
Compressing searchable_911767292_202206_990_2023031021079290.pdf with Ghostscript...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_911767292_202206_990_2023031021079290.pdf
Compressing searchable_911767292_202306_990_2024031222315919.pdf with Ghostscript...
Compressed file saved as form_990_compressed_pdfs/compressed_searchable_911767292_202306_990_2024031222315919.pdf
Compr

### 4. Employee Details - Script for Extracting Funding and Revenue Sources
This script automatically scans all PDFs in a specified folder and extracts funding and revenue details:

2. Script for Extracting Employee and Compensation Details
This script scans all PDFs in the folder to extract employee-related information.